# LeetCode 2877 — Create a DataFrame from List bằng Python (pandas)

Link đề: https://leetcode.com/problems/create-a-dataframe-from-list/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`, chỉ là một hàm; tên hàm / tham số: đối chiếu lại với đề gốc):

```python
import pandas as pd

def createDataframe(student_data: List[List[int]]) -> pd.DataFrame:
    ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **DataFrame** là gì (bảng có **hàng**, **cột** có tên và **chỉ mục `index`**) và biết đổi một **list lồng nhau (2D list)** thành DataFrame.
2. Phân biệt các cách khởi tạo: từ **list các hàng**, từ **dict các cột**, từ **list các dict (records)**, từ **mảng NumPy**.
3. Đi từ cách **vòng lặp thêm từng hàng vào DataFrame rỗng** (chậm, dễ nghĩ) → **list các dict** → **`pd.DataFrame(data, columns=[...])`** (cách chuẩn) → **dict các cột bằng `zip(*data)`** → **`from_records` / NumPy**.
4. Biết so sánh hai DataFrame đúng cách bằng `DataFrame.equals` và `pd.testing.assert_frame_equal`, chú ý **dtype**, **index**, **thứ tự cột**.
5. Tránh các lỗi kinh điển: quên `columns=`, đặt sai thứ tự tên cột, tạo DataFrame **một cột** vì truyền sai dạng dữ liệu, nhầm hàng với cột, dùng `df.append` (đã bị xóa khỏi pandas hiện đại).

## 2. Đề bài & ví dụ

Cho một **list hai chiều** `student_data` chứa mã số và tuổi của một số học sinh: mỗi list con có dạng `[student_id, age]`.

Hãy tạo một **DataFrame** từ `student_data` với **hai cột**, tên lần lượt là `student_id` và `age`, các hàng giữ **cùng thứ tự** như trong list ban đầu.

**Ví dụ**

- Input: `student_data = [[1, 15], [2, 11], [3, 11], [4, 20]]`
- Output:

| student_id | age |
|---|---|
| 1 | 15 |
| 2 | 11 |
| 3 | 11 |
| 4 | 20 |

- Giải thích: một DataFrame được tạo từ `student_data`, có hai cột tên `student_id` và `age`.

**Ràng buộc:** đề không nêu ràng buộc số học đặc biệt; các list con luôn có đúng hai số nguyên. Mọi test trong notebook này dùng dữ liệu hợp lệ như vậy (chi tiết đề: đối chiếu lại với đề gốc).

In [1]:
from typing import List
import pandas as pd
import numpy as np

student_data = [[1, 15], [2, 11], [3, 11], [4, 20]]
print(type(student_data), len(student_data), "hàng; mỗi hàng có", len(student_data[0]), "phần tử")
print("pandas", pd.__version__)

<class 'list'> 4 hàng; mỗi hàng có 2 phần tử
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. DataFrame là gì?

**DataFrame** của pandas giống một **bảng Excel**: có nhiều **hàng** và nhiều **cột**. Mỗi cột có **tên** và có **kiểu dữ liệu (dtype)** riêng. Mỗi hàng có một **nhãn** gọi là **index** (mặc định `0, 1, 2, …`).

Ba "bộ phận" cần nhớ: `df.index` (nhãn hàng), `df.columns` (tên cột), `df.values` (dữ liệu).

In [2]:
df = pd.DataFrame([[1, 15], [2, 11]], columns=["student_id", "age"])
print(df)
print("index  :", list(df.index))
print("columns:", list(df.columns))
print("values :", df.values.tolist())

   student_id  age
0           1   15
1           2   11
index  : [0, 1]
columns: ['student_id', 'age']
values : [[1, 15], [2, 11]]


### 3.2. List hai chiều: "list của các list"

`[[1, 15], [2, 11]]` là một list có 2 phần tử, mỗi phần tử là một list con. Khi đưa vào DataFrame, **mỗi list con là một HÀNG**, và phần tử thứ `t` của list con nằm ở **cột thứ `t`**.

In [3]:
data = [[1, 15], [2, 11], [3, 11]]
print(data[0])          # [1, 15]  -> hàng đầu tiên
print(data[0][1])       # 15       -> hàng 0, cột 1
print(len(data), len(data[0]))      # 3 hàng, 2 cột

[1, 15]
15
3 2


### 3.3. `pd.DataFrame(data, columns=[...])`

Hàm khởi tạo nhận dữ liệu và tùy chọn `columns` (tên cột). Nếu không có `columns`, pandas đặt tên cột là số `0, 1, …`.

In [4]:
print(pd.DataFrame([[1, 15], [2, 11]]))                              # tên cột 0, 1
print(pd.DataFrame([[1, 15], [2, 11]], columns=["student_id", "age"]))   # tên cột theo ý ta

   0   1
0  1  15
1  2  11
   student_id  age
0           1   15
1           2   11


### 3.4. `shape`, `dtypes`, `index`, `columns`

- `df.shape` là tuple `(số hàng, số cột)`.
- `df.dtypes` cho biết kiểu dữ liệu từng cột: số nguyên `int64`, số thực `float64`, chuỗi `str`/`object`…
- `df.index` / `df.columns` là các đối tượng nhãn.

In [5]:
df = pd.DataFrame([[1, 15], [2, 11], [3, 11]], columns=["student_id", "age"])
print(df.shape)            # (3, 2)
print(df.dtypes)
print(df.index)            # RangeIndex(start=0, stop=3, step=1)
print(df.columns.tolist())

(3, 2)
student_id    int64
age           int64
dtype: object
RangeIndex(start=0, stop=3, step=1)
['student_id', 'age']


### 3.5. Tạo từ `dict` các cột

Một cách khác: truyền dict `{tên_cột: list_giá_trị_của_cột}`. Mỗi **khóa** là tên cột, mỗi **giá trị** là **cả một cột** (không phải một hàng!). Các list phải **cùng độ dài**.

In [6]:
by_columns = {"student_id": [1, 2, 3], "age": [15, 11, 11]}
print(pd.DataFrame(by_columns))

   student_id  age
0           1   15
1           2   11
2           3   11


### 3.6. Tạo từ `list các dict` (records)

Mỗi dict là một hàng: khóa = tên cột. Rất tiện khi ta tự dựng dữ liệu bằng vòng lặp.

In [7]:
records = [{"student_id": 1, "age": 15}, {"student_id": 2, "age": 11}]
print(pd.DataFrame(records))

   student_id  age
0           1   15
1           2   11


### 3.7. `zip(*data)`: chuyển hàng thành cột (transpose)

`zip(*data)` "xoay" list các hàng thành các cột: phần tử đầu của mỗi hàng gom vào một nhóm, phần tử thứ hai gom vào nhóm kế…

In [8]:
data = [[1, 15], [2, 11], [3, 11]]
cols = list(zip(*data))
print(cols)                 # [(1, 2, 3), (15, 11, 11)]
print(dict(zip(["student_id", "age"], cols)))

[(1, 2, 3), (15, 11, 11)]
{'student_id': (1, 2, 3), 'age': (15, 11, 11)}


### 3.8. So sánh hai DataFrame

- `a.equals(b)`: trả về `True/False`; yêu cầu **cùng hình dạng, cùng nhãn, cùng dtype từng cột**.
- `pd.testing.assert_frame_equal(a, b)`: nếu khác sẽ **ném lỗi** và mô tả chỗ khác — tiện cho test. Có các tùy chọn `check_dtype`, `check_index_type`...

Không dùng `a == b` trong `assert` trực tiếp, vì nó trả về một **DataFrame các True/False**, không phải một bool.

In [9]:
a = pd.DataFrame([[1, 15]], columns=["student_id", "age"])
b = pd.DataFrame({"student_id": [1], "age": [15]})
print(a.equals(b))                       # True
pd.testing.assert_frame_equal(a, b)      # im lặng nếu bằng nhau
print(a == b)                            # một DataFrame True/False

True
   student_id   age
0        True  True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung bảng cần tạo:

```
          cột "student_id"   cột "age"
hàng 0        1                15        <- student_data[0] = [1, 15]
hàng 1        2                11        <- student_data[1] = [2, 11]
hàng 2        3                11        <- student_data[2] = [3, 11]
hàng 3        4                20        <- student_data[3] = [4, 20]
```

Nhìn là thấy **phép ánh xạ trực tiếp**: mỗi list con của `student_data` đúng bằng một hàng; phần tử thứ nhất là `student_id`, phần tử thứ hai là `age`. Ta chỉ cần "gắn tên cột" `["student_id", "age"]`, theo đúng thứ tự vị trí.

Do đó thuật toán không cần suy luận gì: **một lời gọi khởi tạo** là xong. Việc cần cẩn thận là **thứ tự tên cột** (nếu viết `["age", "student_id"]` thì dữ liệu sẽ bị gán nhầm tên).

## 5. Cách 1 — Brute Force / baseline: thêm từng hàng vào DataFrame rỗng

Cách "tự nhiên" của người mới: tạo DataFrame rỗng có đủ hai cột, rồi duyệt `student_data`, **thêm từng hàng** vào cuối bằng `df.loc[len(df)] = row`.

(Chú ý: phương thức cũ `df.append(...)` đã bị **gỡ bỏ** ở pandas 2.0, nên không dùng được.)

In [10]:
def create_df_append_rows(student_data: List[List[int]]) -> pd.DataFrame:
    df = pd.DataFrame(columns=["student_id", "age"])     # bảng rỗng, đã có tên cột
    for row in student_data:
        df.loc[len(df)] = row                             # thêm hàng có nhãn = số hàng hiện tại
    return df

out = create_df_append_rows([[1, 15], [2, 11], [3, 11], [4, 20]])
print(out)
print(out.dtypes.tolist())
print(out.index)

   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20
[dtype('int64'), dtype('int64')]
Index([0, 1, 2, 3], dtype='int64')


**Độ phức tạp:** mỗi lần thêm hàng, pandas có thể phải **sao chép lại** dữ liệu để mở rộng bảng nên một lần thêm tốn `O(n)`; với `n` hàng tổng **Time `O(n²)`** (xấu nhất), **Space `O(n)`**. Cách này chậm nhất trong các cách dưới đây, cell sau đo thử (số liệu chỉ để quan sát, không ảnh hưởng đúng sai).

**Nút thắt:** DataFrame **không** phải cấu trúc để thêm từng hàng một. Quy tắc vàng của pandas: **dựng dữ liệu bằng Python trước, rồi tạo DataFrame một lần**.

In [11]:
import time
big = [[i, i % 50] for i in range(1500)]

t0 = time.perf_counter(); r1 = create_df_append_rows(big); t1 = time.perf_counter()
r2 = pd.DataFrame(big, columns=["student_id", "age"]); t2 = time.perf_counter()
print(f"thêm từng hàng : {t1 - t0:.3f}s")
print(f"khởi tạo một lần: {t2 - t1:.5f}s")
pd.testing.assert_frame_equal(r1, r2)          # cùng kết quả
print("hai cách cho cùng DataFrame")

thêm từng hàng : 0.602s
khởi tạo một lần: 0.00098s
hai cách cho cùng DataFrame


## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. `pd.DataFrame(...)` **nhận thẳng** một list các list: mỗi list con là một hàng. Không cần duyệt.
2. Tên cột chỉ là tham số `columns=["student_id", "age"]`, gán theo **vị trí**.

Vậy đáp án chỉ một dòng: `pd.DataFrame(student_data, columns=["student_id", "age"])`. Bên trong, pandas làm việc nặng bằng mã C / NumPy, **một lần**, thời gian `O(n)`.

Các cách còn lại (list các dict, dict các cột, NumPy, `from_records`) là những "cánh cửa" khác vào cùng một kết quả; biết chúng giúp ta xử lý đầu vào ở dạng khác trong thực tế.

## 7. Các cách tiếp theo

### Cách 2 — Dựng list các dict (records) bằng vòng lặp Python, rồi tạo DataFrame

**Giải thích từng đoạn.**

- `records = []`: nơi chứa các hàng dạng dict.
- `for sid, age in student_data:` giải nén mỗi list con thành hai biến.
- `records.append({"student_id": sid, "age": age})`: dựng dict `{tên_cột: giá_trị}`.
- `pd.DataFrame(records)`: pandas lấy khóa của dict làm tên cột.

Ưu điểm: tên cột nằm ngay trong code, không thể nhầm thứ tự. Nhược điểm: dài hơn và tốn bộ nhớ vì mỗi hàng là một dict.

In [12]:
def create_df_records(student_data: List[List[int]]) -> pd.DataFrame:
    records = []
    for sid, age in student_data:
        records.append({"student_id": sid, "age": age})
    return pd.DataFrame(records)

print(create_df_records([[1, 15], [2, 11], [3, 11], [4, 20]]))

   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20


In [13]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def createDataframe(student_data: List[List[int]]) -> pd.DataFrame:
    records = [{"student_id": sid, "age": age} for sid, age in student_data]
    return pd.DataFrame(records)

print(createDataframe([[1, 15], [2, 11]]))

   student_id  age
0           1   15
1           2   11


**Độ phức tạp Cách 2:** duyệt `n` hàng `O(n)`, tạo DataFrame `O(n)`. **Time `O(n)`**, **Space `O(n)`** (list các dict). Lưu ý: nếu `student_data` **rỗng**, `pd.DataFrame([])` không có cột nào, nên `shape = (0, 0)`, khác yêu cầu "có hai cột".

---

### Cách 3 — `pd.DataFrame(student_data, columns=[...])` (cách chuẩn)

**Giải thích.** Truyền thẳng list các hàng và tên cột. Với `student_data` rỗng, `pd.DataFrame([], columns=[...])` vẫn giữ **đủ hai cột** (0 hàng).

In [14]:
def create_df_direct(student_data: List[List[int]]) -> pd.DataFrame:
    return pd.DataFrame(student_data, columns=["student_id", "age"])

out = create_df_direct([[1, 15], [2, 11], [3, 11], [4, 20]])
print(out)
print(out.shape, out.dtypes.tolist())
print(create_df_direct([]).shape)      # (0, 2): vẫn có hai cột

   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20
(4, 2) [dtype('int64'), dtype('int64')]
(0, 2)


In [15]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def createDataframe(student_data: List[List[int]]) -> pd.DataFrame:
    return pd.DataFrame(student_data, columns=["student_id", "age"])

print(createDataframe([[1, 15], [2, 11], [3, 11], [4, 20]]))

   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (DataFrame kết quả). Không có vòng lặp Python nào.

---

### Cách 4 — Dựng dict các cột bằng `zip(*data)`

**Giải thích.** `zip(*student_data)` xoay các hàng thành hai nhóm cột. Ghép với tên cột bằng `zip(names, cols)` rồi `dict(...)`. Hữu ích khi dữ liệu thiên về **cột** (mỗi cột xử lý riêng).

In [16]:
def create_df_columns(student_data: List[List[int]]) -> pd.DataFrame:
    names = ["student_id", "age"]
    cols = list(zip(*student_data))                       # [(ids...), (ages...)]
    if not cols:                                          # student_data rỗng
        cols = [(), ()]
    return pd.DataFrame(dict(zip(names, cols)))

print(create_df_columns([[1, 15], [2, 11], [3, 11], [4, 20]]))
print(create_df_columns([]).shape)

   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20
(0, 2)


In [17]:
# Code nộp LeetCode cho Cách 4
import pandas as pd

def createDataframe(student_data: List[List[int]]) -> pd.DataFrame:
    ids = [row[0] for row in student_data]
    ages = [row[1] for row in student_data]
    return pd.DataFrame({"student_id": ids, "age": ages})

print(createDataframe([[1, 15], [2, 11]]))

   student_id  age
0           1   15
1           2   11


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (hai list cột trung gian).

---

### Cách 5 — Mảng NumPy / `DataFrame.from_records`

**Giải thích.**

- `np.array(student_data)` tạo mảng 2 chiều hình `(n, 2)`; `pd.DataFrame(mảng, columns=[...])` biến nó thành bảng. dtype là `int64` (trên Linux/macOS 64-bit; trên Windows cũ NumPy có thể dùng `int32`, vì vậy so sánh dtype cần cẩn thận).
- `pd.DataFrame.from_records(student_data, columns=[...])` là hàm chuyên cho "list các bản ghi (tuple / list)".

In [18]:
arr = np.array([[1, 15], [2, 11], [3, 11], [4, 20]])
print(arr.shape, arr.dtype)
print(pd.DataFrame(arr, columns=["student_id", "age"]))
print(pd.DataFrame.from_records([[1, 15], [2, 11]], columns=["student_id", "age"]))

(4, 2) int64
   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20
   student_id  age
0           1   15
1           2   11


In [19]:
# Code nộp LeetCode cho Cách 5
import pandas as pd

def createDataframe(student_data: List[List[int]]) -> pd.DataFrame:
    return pd.DataFrame.from_records(student_data, columns=["student_id", "age"])

print(createDataframe([[1, 15], [2, 11], [3, 11], [4, 20]]))

   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`**. Ghi chú: nếu mảng NumPy lẫn kiểu (ví dụ cột chuỗi và cột số), `np.array` ép tất cả thành một kiểu; trường hợp đó ưu tiên Cách 3 hoặc Cách 4.

## 8. TRACE từng bước

Trace Cách 1 (thêm từng hàng) để thấy DataFrame **lớn dần** sau mỗi lần thêm, rồi trace Cách 4 (đi qua dict các cột).

In [20]:
def trace_append_rows(student_data):
    df = pd.DataFrame(columns=["student_id", "age"])
    print("BAN ĐẦU (rỗng):", df.shape)
    print(df)
    for step, row in enumerate(student_data, start=1):
        df.loc[len(df)] = row
        print(f"\nBước {step}: thêm hàng {row} với nhãn {len(df) - 1}")
        print(df)
    return df

_ = trace_append_rows([[1, 15], [2, 11], [3, 11]])

BAN ĐẦU (rỗng): (0, 2)
Empty DataFrame
Columns: [student_id, age]
Index: []

Bước 1: thêm hàng [1, 15] với nhãn 0
   student_id  age
0           1   15

Bước 2: thêm hàng [2, 11] với nhãn 1
   student_id  age
0           1   15
1           2   11

Bước 3: thêm hàng [3, 11] với nhãn 2
   student_id  age
0           1   15
1           2   11
2           3   11


In [21]:
def trace_columns(student_data):
    names = ["student_id", "age"]
    cols = list(zip(*student_data))
    print("student_data :", student_data)
    print("zip(*data)   :", cols)
    d = dict(zip(names, cols))
    print("dict các cột :", d)
    df = pd.DataFrame(d)
    print("DataFrame:")
    print(df)
    print("shape =", df.shape, "| dtypes =", df.dtypes.tolist())
    return df

_ = trace_columns([[1, 15], [2, 11], [3, 11]])

student_data : [[1, 15], [2, 11], [3, 11]]
zip(*data)   : [(1, 2, 3), (15, 11, 11)]
dict các cột : {'student_id': (1, 2, 3), 'age': (15, 11, 11)}
DataFrame:
   student_id  age
0           1   15
1           2   11
2           3   11
shape = (3, 2) | dtypes = [dtype('int64'), dtype('int64')]


Bảng tóm tắt trạng thái của Cách 3 với `student_data = [[1, 15], [2, 11]]`:

| bước | biểu thức | kết quả |
|---|---|---|
| 1 | `student_data` | `[[1, 15], [2, 11]]` (2 hàng, mỗi hàng 2 phần tử) |
| 2 | `columns=["student_id", "age"]` | cột 0 → `student_id`, cột 1 → `age` |
| 3 | `pd.DataFrame(...)` | bảng 2×2: hàng 0 = (1, 15), hàng 1 = (2, 11) |
| 4 | `df.shape` | `(2, 2)` |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ là hàm. Trên LeetCode, `import pandas as pd` đã có sẵn trong skeleton; ở notebook này ta tự import `List` để chạy sạch.

In [22]:
import pandas as pd
from typing import List

def createDataframe(student_data: List[List[int]]) -> pd.DataFrame:
    return pd.DataFrame(student_data, columns=["student_id", "age"])

result = createDataframe([[1, 15], [2, 11], [3, 11], [4, 20]])
print(result)
result

   student_id  age
0           1   15
1           2   11
2           3   11
3           4   20


,student_id,age
0,1,15
1,2,11
2,3,11
3,4,20


## 10. Vì sao thuật toán đúng

- **Mỗi hàng ↔ một list con:** quy tắc của `pd.DataFrame` với list các list: phần tử thứ `r` của danh sách ngoài thành hàng có nhãn `r` (index `0, 1, …`), nên **thứ tự hàng được giữ nguyên** như đề yêu cầu.
- **Mỗi vị trí ↔ một cột:** phần tử thứ `t` của mọi list con nằm ở cột thứ `t`. Với `columns=["student_id", "age"]`, cột 0 là `student_id`, cột 1 là `age` — khớp với dạng `[student_id, age]` của đề.
- **Tên cột đúng thứ tự:** tham số `columns` quyết định cả tên và thứ tự, đúng "student_id rồi age".
- **dtype:** mọi giá trị là số nguyên nên pandas suy ra `int64` cho cả hai cột; đây cũng là kiểu mà bộ chấm mong đợi.
- **Trường hợp biên:** với `student_data` rỗng, hàm vẫn trả bảng 0 hàng, 2 cột (nhờ `columns=`), nên không gây lỗi ở chỗ gọi sau.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số học sinh (số hàng).

| Cách | Time | Space |
|---|---|---|
| 1. Thêm từng hàng vào DF rỗng | `O(n²)` (xấu nhất) | `O(n)` |
| 2. List các dict | `O(n)` | `O(n)` (+ n dict nhỏ) |
| 3. `pd.DataFrame(data, columns=)` | `O(n)` | `O(n)` |
| 4. Dict các cột (`zip`) | `O(n)` | `O(n)` (+ list cột) |
| 5. NumPy / `from_records` | `O(n)` | `O(n)` |

Số cột là hằng số (2) nên không xuất hiện trong độ phức tạp. Thực tế Cách 3 nhanh nhất vì không có vòng lặp Python.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Ta so từng cách với một DataFrame **mong đợi dựng tay** bằng `pd.testing.assert_frame_equal` (kiểm cả tên cột, thứ tự cột, dtype và index).

In [23]:
def expected(student_data):
    # dựng "đáp án" độc lập: dict các cột với dtype int64 rõ ràng
    ids = [r[0] for r in student_data]
    ages = [r[1] for r in student_data]
    return pd.DataFrame({"student_id": pd.Series(ids, dtype="int64"),
                         "age": pd.Series(ages, dtype="int64")})

impls = {
    "append_rows": create_df_append_rows,
    "records": create_df_records,
    "direct": create_df_direct,
    "columns": create_df_columns,
    "submit": createDataframe,
}

cases = [
    ("ví dụ của đề", [[1, 15], [2, 11], [3, 11], [4, 20]]),
    ("một hàng", [[7, 30]]),
    ("tuổi trùng nhau", [[1, 10], [2, 10], [3, 10]]),
    ("id không tăng dần", [[9, 18], [2, 19], [5, 20]]),
    ("số lớn", [[10**9, 99], [1, 1]]),
]
for name, data in cases:
    exp = expected(data)
    for key, fn in impls.items():
        got = fn(data)
        pd.testing.assert_frame_equal(got, exp)
        assert list(got.columns) == ["student_id", "age"]
        assert got.shape == (len(data), 2)
    print("PASS", name, "-> shape", exp.shape)

PASS ví dụ của đề -> shape (4, 2)
PASS một hàng -> shape (1, 2)
PASS tuổi trùng nhau -> shape (3, 2)
PASS id không tăng dần -> shape (3, 2)
PASS số lớn -> shape (2, 2)


### 12.2. Hàng cuối khác, thứ tự và dtype

Kiểm các tính chất cụ thể: giữ **thứ tự hàng**, tên cột đúng thứ tự, dtype `int64`, index `0..n-1`.

In [24]:
data = [[4, 20], [3, 11], [2, 11], [1, 15]]
out = createDataframe(data)
assert out["student_id"].tolist() == [4, 3, 2, 1]             # giữ thứ tự hàng
assert out["age"].tolist() == [20, 11, 11, 15]
assert list(out.columns) == ["student_id", "age"]             # đúng tên, đúng thứ tự
assert all(str(t) == "int64" for t in out.dtypes)
assert list(out.index) == [0, 1, 2, 3]
assert out.equals(expected(data))
print("PASS thứ tự / dtype / index")

# Phản ví dụ: đặt tên cột ngược thứ tự thì dữ liệu bị gán nhầm
wrong = pd.DataFrame(data, columns=["age", "student_id"])
assert not wrong.equals(out)
print("tên cột ngược ->", wrong.iloc[0].to_dict(), "(student_id lại là 20!)")

PASS thứ tự / dtype / index
tên cột ngược -> {'age': 4, 'student_id': 20} (student_id lại là 20!)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `n ∈ [1, 200]` hàng `[id, age]` ngẫu nhiên và so sánh **mọi cách** với đáp án mong đợi.

In [25]:
import random
random.seed(2877)
for t in range(200):
    n = random.randint(1, 200)
    data = [[random.randint(1, 10**6), random.randint(1, 100)] for _ in range(n)]
    exp = expected(data)
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(data), exp)
    arr_df = pd.DataFrame(np.array(data, dtype="int64"), columns=["student_id", "age"])
    pd.testing.assert_frame_equal(arr_df, exp)
    fr_df = pd.DataFrame.from_records(data, columns=["student_id", "age"])
    pd.testing.assert_frame_equal(fr_df, exp)
print("200 test ngẫu nhiên: mọi cách khớp DataFrame mong đợi")

200 test ngẫu nhiên: mọi cách khớp DataFrame mong đợi


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `columns=` — tên cột thành `0, 1`

In [26]:
sai = pd.DataFrame([[1, 15], [2, 11]])
print(sai.columns.tolist())                                   # [0, 1]
dung = createDataframe([[1, 15], [2, 11]])
print(dung.columns.tolist())                                  # ['student_id', 'age']
print("bằng nhau?", sai.equals(dung))

[0, 1]
['student_id', 'age']
bằng nhau? False


**Vì sao sai:** đề yêu cầu tên cột **chính xác** `student_id` và `age`. Không đặt `columns` thì pandas đặt `0, 1`, và chấm bài sẽ không khớp tên cột.

### Lỗi 2: Đặt tên cột sai thứ tự

In [27]:
sai = pd.DataFrame([[1, 15], [2, 11]], columns=["age", "student_id"])
print(sai)                                                    # cột "age" chứa 1, 2 (thật ra là id)
print("đúng:")
print(createDataframe([[1, 15], [2, 11]]))

   age  student_id
0    1          15
1    2          11
đúng:
   student_id  age
0           1   15
1           2   11


**Vì sao sai:** tên cột gán theo **vị trí**: phần tử đầu mỗi list con là `student_id`. Đổi chỗ tên làm dữ liệu "trôi" sang cột sai mà **không báo lỗi**. Luôn kiểm bằng cách in vài dòng đầu.

### Lỗi 3: Truyền list phẳng, ra DataFrame một cột

In [28]:
flat = [1, 15, 2, 11]
print(pd.DataFrame(flat, columns=["student_id"]).shape)       # (4, 1): 4 hàng, 1 cột
try:
    pd.DataFrame(flat, columns=["student_id", "age"])
except ValueError as e:
    print("ValueError:", str(e)[:80])

(4, 1)
ValueError: Shape of passed values is (4, 1), indices imply (4, 2)


**Vì sao sai:** dữ liệu phải là list **các hàng** (2 chiều). List phẳng được hiểu là **một cột**. Khi số tên cột không khớp số cột dữ liệu thì pandas báo `ValueError`.

### Lỗi 4: Dùng `df.append` (đã bị xóa) hoặc `concat` trong vòng lặp

In [29]:
print(hasattr(pd.DataFrame, "append"))      # False trong pandas 2.x / 3.x

def sai_concat_trong_vong(student_data):
    df = pd.DataFrame(columns=["student_id", "age"])
    for row in student_data:
        df = pd.concat([df, pd.DataFrame([row], columns=["student_id", "age"])], ignore_index=True)
    return df

out = sai_concat_trong_vong([[1, 15], [2, 11]])
print(out.shape, "-> đúng kết quả nhưng chậm O(n²); nên gom list rồi tạo DataFrame một lần")

False
(2, 2) -> đúng kết quả nhưng chậm O(n²); nên gom list rồi tạo DataFrame một lần


**Vì sao sai / không tốt:** `df.append` không còn. `pd.concat` trong vòng lặp cho ra kết quả đúng nhưng **sao chép toàn bộ bảng ở mỗi vòng**. Hãy gom vào `list` bằng Python rồi gọi `pd.DataFrame` **một lần**.

### Lỗi 5 (nhỏ): So sánh DataFrame bằng `==` trong `assert`

In [30]:
a = pd.DataFrame([[1, 15]], columns=["student_id", "age"])
b = pd.DataFrame([[1, 16]], columns=["student_id", "age"])
try:
    assert a == b                                              # sai: a == b là một DataFrame
except ValueError as e:
    print("ValueError:", str(e)[:70])
print("đúng:", a.equals(b))

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.
đúng: False


**Vì sao sai:** `a == b` trả về DataFrame các `True/False`; chuyển nó thành bool gây `ValueError` ("truth value is ambiguous"). Dùng `equals` hoặc `assert_frame_equal`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "tạo DataFrame từ list / dict" → `pd.DataFrame(data, columns=[...])`. **Một dòng là đủ.**
- Dữ liệu là **list các hàng** → truyền thẳng + `columns=`. Dữ liệu là **dict các cột** → `pd.DataFrame({"tên": [..], ...})`. Dữ liệu là **list các dict** → `pd.DataFrame(records)`.
- **Tránh** thêm từng hàng vào DataFrame trong vòng lặp; gom vào list rồi tạo một lần.
- Luôn kiểm tra nhanh: `df.shape`, `df.dtypes`, `df.head()` sau khi tạo.
- So sánh khi tự test: `pd.testing.assert_frame_equal` (kiểm cả dtype và index).

## 15. Biến thể của bài

1. **Tạo DataFrame từ dict các cột** (ví dụ cột `name`, `age`) và kiểm tra `dtypes`.
2. **Đặt một cột làm index:** `df.set_index("student_id")`.
3. **Thêm cột tính toán** sau khi tạo: LeetCode 2881 *Create a New Column*.
4. **Lấy kích thước bảng:** LeetCode 2878 *Get the Size of a DataFrame*.
5. **Lấy vài hàng đầu:** LeetCode 2879 *Display the First Three Rows*.
6. **Đọc từ CSV** (`pd.read_csv`) thay vì list — cùng kết quả DataFrame.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thêm từng hàng | `df.loc[len(df)] = row` | `O(n²)` | `O(n)` | **tránh**; chỉ để hiểu vì sao chậm |
| 2. List các dict | mỗi hàng một dict | `O(n)` | `O(n)` | dữ liệu sinh bằng vòng lặp, muốn tên cột rõ |
| 3. `pd.DataFrame(data, columns=)` | list các hàng + tên cột | `O(n)` | `O(n)` | **cách nộp bài** |
| 4. Dict các cột | `zip(*data)` hoặc list cột riêng | `O(n)` | `O(n)` | dữ liệu đã tách theo cột |
| 5. NumPy / `from_records` | mảng 2 chiều | `O(n)` | `O(n)` | dữ liệu thuần số; coi chừng dtype |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `pd.DataFrame([[1, 2], [3, 4], [5, 6]]).shape`, `list(pd.DataFrame([[1, 2]], columns=["x", "y"]).columns)`, `pd.DataFrame({"a": [1, 2, 3]}).shape`, `len(pd.DataFrame([[1, 2], [3, 4]]))`.

**Bài 2 (tính tay).** Với `student_data = [[10, 7], [20, 8]]` và `columns=["student_id", "age"]`: vẽ bảng kết quả (nhãn hàng, tên cột, giá trị) và cho biết `shape`.

**Bài 3 (điền TODO).** Hoàn thành `create_todo(student_data)` bằng list các dict.

In [31]:
def create_todo(student_data):
    records = []
    # TODO 1: for sid, age in student_data:
    # TODO 2:     records.append({"student_id": sid, "age": age})
    # TODO 3: return pd.DataFrame(records)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `create_direct(student_data)` bằng **một dòng** `pd.DataFrame(..., columns=[...])` mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `create_with_index(student_data)`: tạo DataFrame như đề nhưng **dùng `student_id` làm index** (và chỉ còn một cột `age`). (b) Viết `create_with_extra(student_data)`: tạo DataFrame rồi thêm cột `is_adult` = `age >= 18` (kiểu bool).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(3, 2)`; `["x", "y"]`; `(3, 1)`; `2`. Bài 2: hai hàng nhãn `0, 1`; hàng 0 = (10, 7), hàng 1 = (20, 8); `shape = (2, 2)`.
- **Mức 2 (vừa):** Bài 3: ba dòng đã ghi trong comment. Bài 4: `return pd.DataFrame(student_data, columns=["student_id", "age"])`.
- **Mức 3 (sâu):** Bài 5(a): dùng `df.set_index("student_id")` trên kết quả của Cách 3. Bài 5(b): `df["is_adult"] = df["age"] >= 18` (so sánh cả cột cho ra cột bool).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert pd.DataFrame([[1, 2], [3, 4], [5, 6]]).shape == (3, 2)
assert list(pd.DataFrame([[1, 2]], columns=["x", "y"]).columns) == ["x", "y"]
assert pd.DataFrame({"a": [1, 2, 3]}).shape == (3, 1)
assert len(pd.DataFrame([[1, 2], [3, 4]])) == 2

# Bài 2
b2 = createDataframe([[10, 7], [20, 8]])
assert b2.shape == (2, 2)
assert b2.loc[0, "student_id"] == 10 and b2.loc[1, "age"] == 8

# Bài 3
def create_todo_ans(student_data):
    records = []
    for sid, age in student_data:
        records.append({"student_id": sid, "age": age})
    return pd.DataFrame(records)

pd.testing.assert_frame_equal(create_todo_ans([[1, 15], [2, 11]]), createDataframe([[1, 15], [2, 11]]))

# Bài 4
def create_direct(student_data):
    return pd.DataFrame(student_data, columns=["student_id", "age"])

pd.testing.assert_frame_equal(create_direct([[1, 15], [2, 11], [3, 11], [4, 20]]),
                              createDataframe([[1, 15], [2, 11], [3, 11], [4, 20]]))

# Bài 5(a)
def create_with_index(student_data):
    return createDataframe(student_data).set_index("student_id")

x = create_with_index([[1, 15], [2, 11]])
assert list(x.columns) == ["age"] and x.index.tolist() == [1, 2]
assert x.loc[2, "age"] == 11

# Bài 5(b)
def create_with_extra(student_data):
    df = createDataframe(student_data)
    df["is_adult"] = df["age"] >= 18
    return df

y = create_with_extra([[1, 15], [2, 11], [3, 20]])
assert y["is_adult"].tolist() == [False, False, True]
assert str(y["is_adult"].dtype) == "bool"
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (list các hàng → DataFrame có tên cột)

```python
import pandas as pd

def createDataframe(student_data: List[List[int]]) -> pd.DataFrame:
    return pd.DataFrame(student_data, columns=["student_id", "age"])
```

**Checklist**

- [ ] Hiểu: mỗi list con = **một hàng**, phần tử thứ `t` = **cột `t`**?
- [ ] Có `columns=["student_id", "age"]` đúng **tên** và đúng **thứ tự**?
- [ ] Không thêm từng hàng trong vòng lặp (`loc`, `concat`, `append`)?
- [ ] Kết quả có `shape = (số hàng, 2)`, dtype số nguyên, index `0..n-1`?
- [ ] Tự test bằng `pd.testing.assert_frame_equal` (không dùng `assert a == b`)?
- [ ] Skeleton đúng `def createDataframe(student_data: List[List[int]]) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?